In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

Siap. Jumlah baris: 600


In [4]:
# Jawaban Soal 1
df.filter(col("metode_pembayaran") == "E-Wallet").select("order_id", "kategori", "total_pendapatan").show(5)

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
+--------+--------------------+----------------+
only showing top 5 rows



In [5]:
# Jawaban Soal 2
df.groupBy("kategori") \
  .agg(spark_sum("total_pendapatan").alias("total_pendapatan_kategori")) \
  .orderBy(col("total_pendapatan_kategori").desc()) \
  .show()

+--------------------+-------------------------+
|            kategori|total_pendapatan_kategori|
+--------------------+-------------------------+
|             Fashion|                124825000|
|          Elektronik|                110600000|
|   Makanan & Minuman|                 95400000|
|Kesehatan & Kecan...|                 82200000|
|        Rumah Tangga|                 77350000|
+--------------------+-------------------------+



In [6]:
# Jawaban Soal 3
df.groupBy("metode_pembayaran").count().show()

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+



In [7]:
from pyspark.sql.functions import when

# Membuat kategori Grosir (jika unit >= 5) dan Eceran (jika < 5)
df_eksplorasi = df.withColumn(
    "tipe_pembelian", 
    when(col("unit_terjual") >= 5, "Grosir").otherwise("Eceran")
)

df_eksplorasi.select("order_id", "unit_terjual", "tipe_pembelian").show(5)

+--------+------------+--------------+
|order_id|unit_terjual|tipe_pembelian|
+--------+------------+--------------+
|ORD-1000|           9|        Grosir|
|ORD-1001|           9|        Grosir|
|ORD-1002|           2|        Eceran|
|ORD-1003|           3|        Eceran|
|ORD-1004|           5|        Grosir|
+--------+------------+--------------+
only showing top 5 rows



In [8]:
# Melihat ringkasan statistik dasar untuk kolom numerik
df.select("unit_terjual", "harga_satuan", "total_pendapatan").describe().show()

+-------+-----------------+------------------+-----------------+
|summary|     unit_terjual|      harga_satuan| total_pendapatan|
+-------+-----------------+------------------+-----------------+
|  count|              600|               600|              600|
|   mean|             4.98|162416.66666666666|817291.6666666666|
| stddev|2.572410444016949|148875.84228560823|947282.1362579074|
|    min|                1|             25000|            25000|
|    max|                9|            500000|          4500000|
+-------+-----------------+------------------+-----------------+



In [9]:
# Memfilter transaksi yang nama kategorinya mengandung kata "Kecantikan"
df.filter(col("kategori").like("%Kecantikan%")).select("order_id", "kategori", "kota").show(5)

+--------+--------------------+----------+
|order_id|            kategori|      kota|
+--------+--------------------+----------+
|ORD-1000|Kesehatan & Kecan...|  Magelang|
|ORD-1007|Kesehatan & Kecan...|      Solo|
|ORD-1011|Kesehatan & Kecan...|  Magelang|
|ORD-1017|Kesehatan & Kecan...| Purworejo|
|ORD-1023|Kesehatan & Kecan...|Yogyakarta|
+--------+--------------------+----------+
only showing top 5 rows

